# Workshop 1: Querying and data integration of genomic databases
**7LMS2137 Databases and Tools · Week 13 · Cystic fibrosis: from a human gene to a bacterial drug target**

Use this notebook together with the workshop document. Each cell has the same label as in the document, for example **[Cell B1]**.

* Run a cell with the ▶ button or **Shift+Enter**. Wait for it to finish (the ▶ stops spinning) before running the next one.
* Words in CAPITALS such as `YOUR_EMAIL` must be replaced before you run the cell.
* Cells starting with `%%bash` run Unix commands, exactly as on the Lecture 2 slides.
* Write your answers on the **answer sheet**, not in this notebook.

> **First:** choose *File → Save a copy in Drive* so that your edits are kept.

## Part 0 – Getting started
**[Cell 0.1]** installs EDirect (NCBI E-utilities), NCBI Datasets and `jq`. It takes 1–2 minutes and prints the EDirect version at the end. If Colab disconnects later in the session, run this cell and Cell 0.2 again.

In [ ]:
# [Cell 0.1] Install the tools (run once per session)
import os
os.environ["PATH"] = "/root/edirect:/usr/local/bin:" + os.environ["PATH"]

!curl -fsSL https://ftp.ncbi.nlm.nih.gov/entrez/entrezdirect/install-edirect.sh -o install-edirect.sh
!yes | bash install-edirect.sh > /dev/null 2>&1
!curl -fsSL -o /usr/local/bin/datasets https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets
!curl -fsSL -o /usr/local/bin/dataformat https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/dataformat
!chmod +x /usr/local/bin/datasets /usr/local/bin/dataformat
!which jq > /dev/null || apt-get -qq install -y jq > /dev/null

!echo "EDirect version:" && esearch -version
!echo "Datasets version:" && datasets --version

**[Cell 0.2]** Replace `YOUR_EMAIL` with your university email address, then run the cell. NCBI asks programmatic users for an email so that it can contact you if a script causes problems.

In [ ]:
# [Cell 0.2] Tell NCBI who you are
%env EMAIL=YOUR_EMAIL

## Part A – Precise queries in the NCBI web interface
Part A is done in your web browser (https://www.ncbi.nlm.nih.gov, database **Nucleotide**). There are no cells to run. Follow Steps A1–A3 in the document and answer **Q1** and **Q2**.

## Part B – Programmatic retrieval with EDirect
Reminder: `esearch` → IDs · `esummary` → short summaries · `efetch` → full records · `elink` → related records in another database · `xtract` → turns XML into a table (`-pattern` = rows, `-element` = columns).

**[Cell B1]** Human *CFTR* in NCBI Gene → **Q3**

In [ ]:
%%bash
# [Cell B1] Gene ID, location, description and coordinates of human CFTR
esearch -db gene -query "CFTR[GENE] AND Homo sapiens[ORGN] AND alive[PROP]" |
  esummary |
  xtract -pattern DocumentSummary -element Id Name MapLocation Description \
         -block GenomicInfoType -element ChrAccVer ChrStart ChrStop

**[Cell B2]** The RefSeq Select mRNA of *CFTR*, saved as FASTA → **Q4**

In [ ]:
%%bash
# [Cell B2] Download the RefSeq Select transcript, show its header and count its nucleotides
esearch -db nuccore \
  -query "CFTR[GENE] AND Homo sapiens[ORGN] AND biomol_mrna[PROP]" |
  efilter -source select |
  efetch -format fasta > CFTR_select.fasta

head -n 1 CFTR_select.fasta
grep -v ">" CFTR_select.fasta | tr -d "\n" | wc -c

**[Cell B3]** Links from the *CFTR* gene record to OMIM, ClinVar and PubMed → **Q5**

In [ ]:
%%bash
# [Cell B3] Count the linked records in three other databases
for target in omim clinvar pubmed
do
  n=$(esearch -db gene \
        -query "CFTR[GENE] AND Homo sapiens[ORGN] AND alive[PROP]" |
      elink -target $target |
      xtract -pattern ENTREZ_DIRECT -element Count)
  echo "$target: $n"
  sleep 1
done

### Thread B: switching to the pathogen
**[Cell B4]** Complete chromosomes of *Pseudomonas aeruginosa* in RefSeq → **Q6(a)**

In [ ]:
%%bash
# [Cell B4] Count complete, circular RefSeq sequences that are not plasmids
query='"Pseudomonas aeruginosa"[ORGN] AND srcdb_refseq[PROP]'
query="$query AND topology_circular[PROP] AND completeness_complete[PROP]"
query="$query NOT plasmid[TITL]"

esearch -db nuccore -query "$query" |
  xtract -pattern ENTREZ_DIRECT -element Count

**[Cell B5]** Your turn → **Q6(b)**. This is a copy of Cell B1. Edit it so that it finds *gyrA* in *P. aeruginosa* PAO1:
1. Replace `CFTR[GENE] AND Homo sapiens[ORGN]` with `gyrA[GENE] AND txid208964[ORGN]` (208964 is the NCBI Taxonomy ID of strain PAO1). Keep `alive[PROP]`.
2. Add `OtherAliases` after `Description`, so that the locus tag is printed too.

In [ ]:
%%bash
# [Cell B5] EDIT this copy of Cell B1 (see the instructions above)
esearch -db gene -query "CFTR[GENE] AND Homo sapiens[ORGN] AND alive[PROP]" |
  esummary |
  xtract -pattern DocumentSummary -element Id Name MapLocation Description \
         -block GenomicInfoType -element ChrAccVer ChrStart ChrStop

## Part C – Data integration: reconciling identifiers
`curl` sends a request to a REST API; `jq` picks fields out of the JSON answer (Lecture 2, slides 31–33 and 40).

**[Cell C1]** *CFTR* in Ensembl → **Q7**

In [ ]:
%%bash
# [Cell C1] Look up CFTR by its symbol in the Ensembl REST API
curl -s 'https://rest.ensembl.org/lookup/symbol/homo_sapiens/CFTR' \
     -H 'Content-type:application/json' |
  jq '{id, version, seq_region_name, start, end,
       assembly_name, canonical_transcript}'

**[Cell C2]** Replace `ENSEMBL_GENE_ID` with the Ensembl gene ID from Q7 **without** its version number (no `.20` at the end), then run → **Q8**

In [ ]:
%%bash
# [Cell C2] Cross-references (xrefs) of the gene in five databases
curl -s 'https://rest.ensembl.org/xrefs/id/ENSEMBL_GENE_ID' \
     -H 'Content-type:application/json' |
  jq -r '.[] | [.dbname, .primary_id, .description] | @tsv' |
  grep -E 'HGNC|EntrezGene|MIM_GENE|MIM_MORBID|Uniprot_gn'

Steps **C3** (UniProt ID mapping), **C4** (ClinVar) and **C5** (UniProt and OMA for GyrA) are done in your web browser. Follow the document and answer **Q8–Q10**.

## Part D – Bridging the two threads
**[Cell D1]** Papers from 2025 about both cystic fibrosis and *P. aeruginosa* (MeSH headings) → **Q11**

In [ ]:
%%bash
# [Cell D1] Count the papers, then show the first three as a table
query='"Cystic Fibrosis"[MH] AND "Pseudomonas aeruginosa"[MH] AND 2025[PDAT]'

esearch -db pubmed -query "$query" |
  xtract -pattern ENTREZ_DIRECT -element Count

esearch -db pubmed -query "$query" | esummary |
  xtract -pattern DocumentSummary -element Id PubDate Title | head -n 3

Finish with **Q12** on your answer sheet.

## Optional extension
**[Cell E1]** *CFTR* orthologues in mammals with NCBI Datasets. How many species are listed? Would a similar command find a *gyrA* orthologue in humans?

In [ ]:
%%bash
# [Cell E1] List mammalian orthologues of human CFTR
datasets summary gene symbol CFTR --taxon human \
  --ortholog mammals --as-json-lines |
  dataformat tsv gene --fields tax-name,symbol,gene-id | head -n 20